# vLLM Optima: Colab quickstart

**Self-healing, hardware-aware optimizer for LLM serving.** Give it any model: it fixes what breaks, makes it faster, and proves every change on your GPU.

Reasoning: **NVIDIA Nemotron** on **Nebius Token Factory** · Web search: **Tavily** · Engine: **vLLM 0.31.0**

### Before you start
1. **GPU:** *Runtime → Change runtime type → T4 GPU* (the setup all 30 tests ran on).
2. **Keys:** open **Secrets** (🔑 in the left sidebar) and add:
   - `NEBIUS_API_KEY`: required ([Nebius Token Factory](https://tokenfactory.nebius.com))
   - `TAVILY_API_KEY`: optional; without it, web search is simply off
3. Run the 4 cells below in order. **First run: ~10 minutes** (installing vLLM).

> No GPU or key? Watch the 29 recorded runs in the live demo instead: [LINK_TO_HF_SPACE](LINK_TO_HF_SPACE)

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader || echo "No GPU found: choose Runtime -> Change runtime type -> T4 GPU, then run this cell again."

## 2 · Get the code and install vLLM
vLLM goes into its own environment (`/content/vllm-env`) so it can't clash with Colab's packages. Skipped if it's already installed.

In [ ]:
import os
if not os.path.exists("/content/vllm-optima"):
    !git clone -q https://github.com/Raghu-dev-pixel/vllm-optima /content/vllm-optima
if not os.path.exists("/content/vllm-env/bin/python"):
    !pip install -q uv
    !uv venv -q /content/vllm-env --python 3.12
    !uv pip install -q --python /content/vllm-env/bin/python vllm==0.31.0
!pip install -q -r /content/vllm-optima/app/requirements.txt
print("Ready.")

## 3 · Load your keys from Colab Secrets
Keys are read from Secrets only: never paste them into the notebook.

In [ ]:
import os
from google.colab import userdata

os.environ["NEBIUS_API_KEY"] = userdata.get("NEBIUS_API_KEY")
try:
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
    print("Nebius + Tavily keys loaded.")
except Exception:
    print("Nebius key loaded. No Tavily key: web search is off.")

## 4 · Start the app
Click the **public URL** (`https://….gradio.live`) that appears below.

**Things to try**
- Beginner mode: `Qwen/Qwen2.5-0.5B-Instruct` → works first time, then press **⚡ Optimize for this GPU**
- A config copied from an H100: model `Qwen/Qwen2.5-0.5B-Instruct`, settings `{"dtype": "bfloat16", "kv_cache_dtype": "fp8", "tensor_parallel_size": 2}` → three fixes, verified
- Too big for a T4: `Qwen/Qwen2.5-7B-Instruct` → switches to the 4-bit AWQ variant
- The **Recorded runs** tab replays all 29 test runs

This cell keeps running while the app is open. Stop it to shut the app down.

In [ ]:
%cd /content/vllm-optima/app
import app
app.demo.queue().launch(share=True, debug=True)